In [1]:
import requests
import time
import pandas as pd
from sql_db_func import MySQLConnetFunc
from datetime import datetime, timedelta
dbhandler = MySQLConnetFunc('l6a01_project')

In [2]:
now = datetime.now()
ym = now.strftime("%Y%m")
print(f"連線時間: {now}({ym})")
one_mon_ago = now - timedelta(days=30)

連線時間: 2025-11-17 08:05:46.344122(202511)


In [ ]:
class Config:
    def __init__(self):
        # ================================================= datetime =================================================
        #now = datetime.now()
        #ym = now.strftime("%Y%m")
        #print(f"連線時間: {now}({ym})")
        #one_mon_ago = now - timedelta(days=30)

        # ================================================= 資料設定  =================================================
        # === PI Line & AOI Tool  ====
        self.uni_aoi_names = [f'aoi{i}00' for i in range(1,4,1)]
        self.uni_pi_names = [f'pi{i}00' for i in range(1,8,1)]

        # === particle type & defect code ====
        self.uni_UPI_defect_codes = ['Polymer', 'SSIU_Polymer']
        self.uni_SPOT_defect_codes = ['PI_Spot_NP', 'PIS With Particle']
        self.uni_SPS_defect_codes = ['SPS']
        self.uni_defect_types = ['Particle', 'PISpot']
        self.all_defect_codes = ['Polymer', 'SSIU_Polymer', 'PI_Spot_NP', 'PIS With Particle', 'SPS']

        # ==== defect size config  ===
        self.uni_defect_sizes = ['S', 'M', 'L', 'O']
        self.rawdata_defect_size_col = 'defect_size'
        self.size_group_keys = ['S','M','L','O','SM','SL','SO','ML','MO','LO','SML','SMO','SLO','MLO','SMLO']
        self.defect_size_rules = {
            "S": lambda x: x <= 20,
            "M": lambda x: 21 <= x <= 100,
            "L": lambda x: 101 <= x <= 400,
            "O": lambda x: x >= 401,
        }

        # === Glass side  ===
        self.glass_sides = ['CF', 'TFT']

        # =================================================  UPI/SPOT Default config  =================================================
        self.tab_filter_config = {
            'UPI': {
                'line_id': ['CAPIC200'],
                'ai_code_1': self.uni_UPI_defect_codes,
                'recipe_id':[]
            },
            'PISpot': {
                'line_id': ['CAPIC200'],
                'ai_code_1': self.uni_SPOT_defect_codes,
                'recipe_id':[]
            },
            'SPS': {
                'line_id': [f'CAPIC{i}00' for i in range(1,8,1)],
                'ai_code_1': self.uni_SPS_defect_codes,
                'recipe_id':[]
            },
            'SPEC':{}
        }

        # ================================================= MySQL AOI Density 資料表設定 =================================================
        self.aoi_pidensit_summary_tbn = f'pidenisty_pihour_{ym}'
        self.aoi_pi_density_tbns = [f'{aoi}_pidensity_yyyymm_{pi}' for aoi in self.uni_aoi_names for pi in self.uni_pi_names]
        self.aoi_pidensity_spec_tbn = 'aoi_spec_for_aoimonitor'
        print('預設讀取:', self.aoi_pidensit_summary_tbn)

        self.aoi_pidensity_spec_cols = ['NO', 'MODEL_ID', 'MODEL_TYPE', 'DEFECT_CODE', 'PROCESS_TYPE', 'SIZE_TYPE', 'OOC', 'OOS']
        self.aoi_density_summary_sql_cols = [
            'scan_time', 'pi_time', 'pi_hour', 'aoi', 'line_id', 'model',
            'glass_type', 'recipe_id', 'ai_code_1', 'pic_paths',
            'maingroup_glass_count', 'maingroup_defect_count',
            'defect_code_glass_count', 'defect_code_count',
            'small_defect_count', 'middle_defect_count', 'large_defect_count', 'over_defect_count',
            'glass', 'noteText'
        ]

        self.aoi_density_rawdata_sql_cols = [
            'scan_time', 'line_id', 'model', 'glass_type', 'recipe_id', 'glass_id',
            'pic_name', 'x', 'y', 'predict_code', 'judge_code', 'mark', 'hour','dayhour', 'day', 'year', 'month', 'season', 'week', 'yearmonth',
            'defect_count', 'defect_size', 'open_status', 'ai_code_1', 'ai_code_2',
            'ai_code_3', 'ai_code_4', 'ai_code_5', 'gray_name', 'ip_num',
            'first_code', 'chip_name', 'defect_seq', 'pi_time', 'pi_hour',
            'pic_path', 'recipe_comment'
        ]

        
        self.PRIMARY_GROUP_COLS = ["pi_hour", "line_id", "model", "glass_type", "recipe_id"]
        # ================================================= 前端網頁CONFIG =================================================
        self.chart_table_coldict = {
            'line_id': 'PI Line',
            'aoi': 'aoi',
            'model': 'Model',
            'glass_type': 'side',
            'pi_hour': 'Hourly',
            'recipe_id': 'recipe',
            'maingroup_glass_count': 'total gld',   # maingroup分群後的玻璃總片數
            'ai_code_1': 'defect',
            #'maingroup_defect_count': 'total def',  # maingroup分群後的總defect 數
            'glass': 'glass',                        # list ,次分群中對應的glass名稱
            #'defect_code_glass_count': 'gld',  # 主群後再分ai_code_1的玻璃數
            'defect_code_count': 'def count ',        # 主群後再分ai_code_1的defect數
            
            'glass_defect_count': 'size' # dsict, 單片glass defect
        }

        self.table_group_key_dict = {
            'main_group': ['pi_hour', 'line_id', 'aoi', 'model', 'recipe_id', 'glass_type', 'maingroup_glass_count', 'ai_code_1'],
                           #'maingroup_glass_count', 'maingroup_defect_count'],
            'uni_col': ['defect_code_glass_count', 'defect_code_count', 'glass_defect_count', 'glass']
            # glass 依照現有邏輯分割後單獨對應 uni_col 中的其他欄位資訊
        }

        self.chart_group_dict = {
            'left': ['line_id','model','maingroup_glass_count', 'defect_code_glass_count'],
            'up':   ['aoi', 'ai_code_1'],
            'down': ['pi_hour'],
            'right':['density'],
        }

        self.uni_glass_row_info_dict = {
            'glass_id': 'glass',
            'glass_defect_count':'glass_defect_count',
            'small_defect_count': 'S',
            'middle_defect_count': 'M',
            'large_defect_count': 'L',
            'over_defect_count':  'O'
        }

        self.defect_group_coldict = {'x': 'x', 'y': 'y', 'chip_name': 'chip', 'pic_name':'img'}

        self.filter_item_coldict = {
            'line_id': 'PI Line',
            'aoi': 'aoi tools',
            'model': 'Model',
            'recipe_id': 'recipe',
            'glass_type': 'glass_side',
            'ai_code_1': 'defect code',
            'defect_size': 'defect size'
        }

        self.front_config = {
            'chartKeyDict': self.chart_group_dict,
            'filtetItemKeyDict': self.filter_item_coldict,
            'hourlyTable': self.chart_table_coldict,
            'hourlyTable_key_group': self.table_group_key_dict,
            'uniGlassInfo': self.uni_glass_row_info_dict,
            'uniGlassDefectTable': self.defect_group_coldict,
            'SubTabsFilterDefaultDict': self.tab_filter_config
        }

In [3]:

def datamall_get(url):
    Header={"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.157 Safari/537.36"}
    headers={"Content-Type":"application/json"}
    proxyDict={
        "http":"http://10.97.4.1:8080",
        "https":"http://10.97.4.1:8080"
    }
    proxies=proxyDict
    #網址由"複製自取訂單網址"按鈕取得
    N = 0
    while True : 
        if N > 6 :
            break
        try :
            r = requests.get(url,proxies=proxyDict)
            #r = requests.get(url)
            data = r.json()
            break
        except :
            print("retry  ....  " )
            time.sleep(30)
            N += 1
        
    return data


In [ ]:
df = dbhandler

In [ ]:
tbns = dbhandler.list_tables()
tbns

drops = ['aoi100_glassdata_202509',
 'aoi100_glassdata_202510',
 'aoi100_glassdata_202511',
 
 'aoi200_glassdata_202509',
 'aoi200_glassdata_202510',
 'aoi200_glassdata_202511',
 
 
 'aoi300_glassdata_202511',
 
 
 'his_90days_spec_table',
 
 'inspection_summary_table',
 
 
 ]

#for tbn in drops :
#    dbhandler.drop_table(tbn)

2025-11-17 08:58:28,983 - INFO - [list_tables] 成功取得資料表名稱，共 90 張表。
2025-11-17 08:58:29,426 - INFO - [drop_table] 資料表 'aoi100_glassdata_202509' 已刪除.
2025-11-17 08:58:29,746 - INFO - [drop_table] 資料表 'aoi100_glassdata_202510' 已刪除.
2025-11-17 08:58:30,119 - INFO - [drop_table] 資料表 'aoi100_glassdata_202511' 已刪除.
2025-11-17 08:58:30,401 - INFO - [drop_table] 資料表 'aoi200_glassdata_202509' 已刪除.
2025-11-17 08:58:30,682 - INFO - [drop_table] 資料表 'aoi200_glassdata_202510' 已刪除.
2025-11-17 08:58:30,981 - INFO - [drop_table] 資料表 'aoi200_glassdata_202511' 已刪除.
2025-11-17 08:58:31,363 - INFO - [drop_table] 資料表 'aoi300_glassdata_202511' 已刪除.
2025-11-17 08:58:31,682 - INFO - [drop_table] 資料表 'his_90days_spec_table' 已刪除.
2025-11-17 08:58:31,988 - INFO - [drop_table] 資料表 'inspection_summary_table' 已刪除.


In [17]:
url = 'http://tcpaaie101.corpnet.auo.com:8005/api/datamall/eyJrZXkiOiAiMTcwZWNhZWMxNzAyZmJiZmQ4ZDljMTA3Y2U2YzI3NTQiLCAiaWQiOiAiMjAyMzA5MDYxNzUyMTU3MTE0MyJ9'
summary_cols = ['CHIP_COUNT', 'CHIP_JUDGE', 'CHIP_OK_COUNT', 'DEFECT', 'FAB','MODEL_NO', 'RECIPE_NAME', 'RUN_ID', 'SCAN_ENDTIME', 'SCAN_STARTTIME',
                'SHEET_ID', 'STAGE', 'TOOL_ID', 'TOTAL_DEFECT_COUNT', 'TYPE']
summary_tbn = 'inspection_summary_table'

datas = datamall_get(url)
df=pd.DataFrame(datas['1'])

print('data mall row:', len(df))
if not df.empty:
    dbhandler.append_new_rows( df, summary_tbn, summary_cols)

print(f'now {summary_tbn} row: ', len(dbhandler.get_table(summary_tbn)))

2025-11-14 14:55:14,681 - INFO - [append_new_rows] 無新增資料可寫入 'inspection_summary_table'.


data mall row: 22
now inspection_summary_table row:  130


In [6]:
"""
將raw_tbn根據當下撈取時間存入特定月份的資料表,raw_tbn資料表名稱格式改為f'inspection_raw_table_{yyyymm}'

"""
url = 'http://tcpaaie101.corpnet.auo.com:8005/api/datamall/eyJrZXkiOiAiMjcwOWJlNzc4ZWNmOWZhMGEzYjc2MjBjN2MzNjMwN2YiLCAiaWQiOiAiMjAyMzA5MDYxNzUyNDg3MTE0NCJ9'
#summary_cols = ['CHIP_COUNT', 'CHIP_JUDGE', 'CHIP_OK_COUNT', 'DEFECT', 'FAB','MODEL_NO', 'RECIPE_NAME', 'RUN_ID', 'SCAN_ENDTIME', 'SCAN_STARTTIME',
#                'SHEET_ID', 'STAGE', 'TOOL_ID', 'TOTAL_DEFECT_COUNT', 'TYPE']
raw_tbn = f'inspection_raw_table_{ym}'
raw_cols = ['CHIP_COUNT', 'CHIP_JUDGE', 'CHIP_OK_COUNT', 'DEFECT', 'FAB','MODEL_NO', 'RECIPE_NAME', 'RUN_ID', 'SCAN_ENDTIME', 'SCAN_STARTTIME','SHEET_ID', 'STAGE', 'TOOL_ID', 'TOTAL_DEFECT_COUNT', 'TYPE']

datas = datamall_get(url)
df=pd.DataFrame(datas['1'])


print('data mall row:', len(df))
if not df.empty:
    dbhandler.append_new_rows( df, raw_tbn, raw_cols)

print(f'now {raw_tbn} row: ', len(dbhandler.get_table(raw_tbn)))

2025-11-14 15:36:17,879 - WARNING - [append_new_rows] 無法取得 'inspection_raw_table_202511'，直接新建。


data mall row: 12


2025-11-14 15:36:20,060 - INFO - [append_new_rows] 'inspection_raw_table_202511' 已建立，新增 12 筆資料.


now inspection_raw_table_202511 row:  12


In [7]:
df.columns

Index(['COORD_X', 'COORD_Y', 'DEFECT', 'DEFECT_AREA', 'DEFECT_ID',
       'DEFECT_SIZE_TYPE', 'FAB', 'FRONT_REVERSE', 'IMG_URL', 'RECIPE_NAME',
       'RUN_ID', 'SCAN_ENDTIME', 'SCAN_STARTTIME', 'SHEET_ID', 'SP', 'STAGE',
       'TOOL_ID', 'TOTAL_DEFECT_COUNT'],
      dtype='object')